# CatBoost Term Classification V2

This notebook trains CatBoost models using **MultiOutputClassifier**.

**Why CatBoost?**
- Often **best overall performance**
- Handles class imbalance automatically
- Robust to overfitting
- Good default parameters

**Sections:**
1. CogAtlas data (no PCA)
2. CogAtlas data (with PCA)
3. Hierarchical data (no PCA)
4. Hierarchical data (with PCA)
5. Hyperparameter tuning
6. Model comparison
7. Save/load models

In [1]:
import sys
sys.path.append('../../')

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import seaborn as sns

from neurovlm.term_classification_models import (
    load_and_align_term_data,
    CatBoostTermPredictor,
    evaluate_term_prediction,
    print_term_evaluation_results,
    save_term_results
)

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 1. CogAtlas Data (No PCA)

In [ ]:
# Load data
print("Loading CogAtlas term data...")
X, y, pmids, term_names = load_and_align_term_data(
    data_source="cogatlas"
)

print(f"\nDataset shape: {X.shape}")
print(f"Labels shape: {y.shape}")

In [3]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Train: {X_train.shape}")
print(f"Test: {X_test.shape}")

Train: (23894, 384)
Test: (5974, 384)


In [4]:
# Train CatBoost (no PCA)
print("\nTraining CatBoost (no PCA)...")
cat_model = CatBoostTermPredictor(
    iterations=100,
    depth=3,
    learning_rate=0.1,
    l2_leaf_reg=3.0,
    subsample=0.8,
    random_state=42,
    verbose=False,
    n_jobs=-1
)

cat_model.fit(X_train, y_train)


Training CatBoost (no PCA)...
Scaling features...
Training CatBoost multi-output classifier...
  Input shape: (23894, 384)
  Output shape: (23894, 808)
Training complete!


In [5]:
# Evaluate
print("\nEvaluating...")
y_pred_proba = cat_model.predict_proba(X_test)
y_pred = cat_model.predict(X_test, threshold=0.5)

results_cat_no_pca = evaluate_term_prediction(
    y_test, y_pred, y_pred_proba, term_names, top_k=10
)

print_term_evaluation_results(results_cat_no_pca)


Evaluating...

=== Overall Performance ===
Metric                              Micro      Macro
----------------------------------------------------
Precision                           0.832      0.663
Recall                              0.329      0.216
F1                                  0.471      0.307

=== Ranking Metrics ===
  label_ranking_avg_precision: 0.6815
  coverage_error: 84.8021
  label_ranking_loss: 0.0165

=== Top-10 Accuracy ===
  1.000 - Fraction of samples with at least one true term in top-10

=== Recall@K Metrics ===
  recall@5: 0.3350 - Average fraction of true labels found in top-5
  recall@7: 0.4355 - Average fraction of true labels found in top-7
  recall@10: 0.5538 - Average fraction of true labels found in top-10

=== AUC Metrics ===
  Micro-AUC: 0.9829
  Macro-AUC: 0.9756

=== Performance on Most Common Terms ===
Term                                      Precision     Recall         F1    Support
------------------------------------------------------------

## 2. CogAtlas Data (With PCA)

In [12]:
# Apply PCA
print("Applying PCA...")
pca = PCA(n_components=0.95, random_state=42)
X_train_pca = pca.fit_transform(X_train)
X_test_pca = pca.transform(X_test)

print(f"Explained variance: {pca.explained_variance_ratio_.sum():.2%}")
print(f"Shape after PCA: {X_train_pca.shape}")

Applying PCA...
Explained variance: 95.06%
Shape after PCA: (23894, 119)


In [13]:
# Train CatBoost with PCA
print("\nTraining CatBoost (with PCA)...")
cat_model_pca = CatBoostTermPredictor(
    iterations=100,
    depth=3,
    learning_rate=0.1,
    l2_leaf_reg=3.0,
    subsample=0.8,
    random_state=42,
    verbose=False,
    n_jobs=-1
)

cat_model_pca.fit(X_train_pca, y_train)


Training CatBoost (with PCA)...
Scaling features...
Training CatBoost multi-output classifier...
  Input shape: (23894, 119)
  Output shape: (23894, 808)
Training complete!


In [14]:
# Evaluate
print("\nEvaluating...")
y_pred_proba_pca = cat_model_pca.predict_proba(X_test_pca)
y_pred_pca = cat_model_pca.predict(X_test_pca, threshold=0.5)

results_cat_pca = evaluate_term_prediction(
    y_test, y_pred_pca, y_pred_proba_pca, term_names, top_k=10
)

print_term_evaluation_results(results_cat_pca)


Evaluating...

=== Overall Performance ===
Metric                              Micro      Macro
----------------------------------------------------
Precision                           0.860      0.655
Recall                              0.245      0.156
F1                                  0.381      0.232

=== Ranking Metrics ===
  label_ranking_avg_precision: 0.6329
  coverage_error: 104.1815
  label_ranking_loss: 0.0216

=== Top-10 Accuracy ===
  0.999 - Fraction of samples with at least one true term in top-10

=== Recall@K Metrics ===
  recall@5: 0.3172 - Average fraction of true labels found in top-5
  recall@7: 0.4092 - Average fraction of true labels found in top-7
  recall@10: 0.5194 - Average fraction of true labels found in top-10

=== AUC Metrics ===
  Micro-AUC: 0.9778
  Macro-AUC: 0.9668

=== Performance on Most Common Terms ===
Term                                      Precision     Recall         F1    Support
-----------------------------------------------------------

## 3. Hierarchical Data (Intermediate - No PCA)

In [ ]:
# Load hierarchical data
print("Loading intermediate hierarchical term data...")
X_hier, y_hier, pmids_hier, term_names_hier = load_and_align_term_data(
    data_source='fine_hierarchical_term'
)

print(f"\nHierarchical dataset shape: {X_hier.shape}")
print(f"Labels shape: {y_hier.shape}")

In [16]:
# Split
X_train_hier, X_test_hier, y_train_hier, y_test_hier = train_test_split(
    X_hier, y_hier, test_size=0.2, random_state=42
)

# Train
print("\nTraining CatBoost on hierarchical data (no PCA)...")
cat_model_hier = CatBoostTermPredictor(
    iterations=100,
    depth=3,
    learning_rate=0.1,
    l2_leaf_reg=3.0,
    subsample=0.8,
    random_state=42,
    verbose=False,
    n_jobs=-1
)

cat_model_hier.fit(X_train_hier, y_train_hier)


Training CatBoost on hierarchical data (no PCA)...
Scaling features...
Training CatBoost multi-output classifier...
  Input shape: (23894, 384)
  Output shape: (23894, 877)
Training complete!


In [17]:
# Evaluate
print("\nEvaluating...")
y_pred_proba_hier = cat_model_hier.predict_proba(X_test_hier)
y_pred_hier = cat_model_hier.predict(X_test_hier, threshold=0.5)

results_cat_hier = evaluate_term_prediction(
    y_test_hier, y_pred_hier, y_pred_proba_hier, term_names_hier, top_k=10
)

print_term_evaluation_results(results_cat_hier)


Evaluating...

=== Overall Performance ===
Metric                              Micro      Macro
----------------------------------------------------
Precision                           0.832      0.611
Recall                              0.250      0.199
F1                                  0.384      0.283

=== Ranking Metrics ===
  label_ranking_avg_precision: 0.5835
  coverage_error: 143.6861
  label_ranking_loss: 0.0282

=== Top-10 Accuracy ===
  0.999 - Fraction of samples with at least one true term in top-10

=== Recall@K Metrics ===
  recall@5: 0.2605 - Average fraction of true labels found in top-5
  recall@7: 0.3364 - Average fraction of true labels found in top-7
  recall@10: 0.4244 - Average fraction of true labels found in top-10

=== AUC Metrics ===
  Micro-AUC: 0.9708
  Macro-AUC: 0.9381

=== Performance on Most Common Terms ===
Term                                      Precision     Recall         F1    Support
-----------------------------------------------------------

## 4. Hierarchical Data (With PCA)

In [ ]:
# Apply PCA to hierarchical data
print("Applying PCA to hierarchical data...")
pca_hier = PCA(n_components=256, random_state=42)
X_train_hier_pca = pca_hier.fit_transform(X_train_hier)
X_test_hier_pca = pca_hier.transform(X_test_hier)

print(f"Explained variance: {pca_hier.explained_variance_ratio_.sum():.2%}")

# Train
print("\nTraining CatBoost on hierarchical data (with PCA)...")
cat_model_hier_pca = CatBoostTermPredictor(
    iterations=100,
    depth=3,
    learning_rate=0.1,
    l2_leaf_reg=3.0,
    subsample=0.8,
    random_state=42,
    verbose=False,
    n_jobs=-1
)

cat_model_hier_pca.fit(X_train_hier_pca, y_train_hier)

In [ ]:
# Evaluate
print("\nEvaluating...")
y_pred_proba_hier_pca = cat_model_hier_pca.predict_proba(X_test_hier_pca)
y_pred_hier_pca = cat_model_hier_pca.predict(X_test_hier_pca, threshold=0.5)

results_cat_hier_pca = evaluate_term_prediction(
    y_test_hier, y_pred_hier_pca, y_pred_proba_hier_pca, term_names_hier, top_k=10
)

print_term_evaluation_results(results_cat_hier_pca)

## 5. Hyperparameter Tuning

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import make_scorer

print("Setting up hyperparameter search...")

# Define custom Recall@K scorer for ranking quality
def recall_at_k(y_true, y_pred_proba, k=10):
    """Calculate mean Recall@K across all samples.

    This metric optimizes for retrieval quality - getting as many
    true labels as possible into the top-K predictions.
    """
    recalls = []
    for i in range(len(y_true)):
        top_k_indices = np.argsort(y_pred_proba[i])[-k:]
        true_indices = np.where(y_true[i] == 1)[0]
        if len(true_indices) > 0:
            hits = np.sum(np.isin(top_k_indices, true_indices))
            recalls.append(hits / len(true_indices))
    return np.mean(recalls) if recalls else 0.0

# Create scorer that works with predict_proba
recall_at_10_scorer = make_scorer(
    recall_at_k,
    greater_is_better=True,
    needs_proba=True,
    k=10
)

print("Using Recall@10 as optimization metric (optimizes for top-10 retrieval quality)")

# Define parameter distributions (CatBoost-specific)
param_dist = {
    'iterations': [50, 100, 200],
    'depth': [2, 3, 4, 5, 6],
    'learning_rate': [0.01, 0.05, 0.1],
    'l2_leaf_reg': [1.0, 3.0, 5.0, 7.0],
    'subsample': [0.6, 0.8, 1.0]
}

# Create search object with Recall@10 scoring
search = RandomizedSearchCV(
    CatBoostTermPredictor(random_state=42, verbose=False, n_jobs=1),
    param_dist,
    n_iter=20,
    cv=3,
    scoring=recall_at_10_scorer,  # Optimizes for top-10 retrieval
    random_state=42,
    n_jobs=-1,
    verbose=2
)

print("\nStarting hyperparameter search (this may take a while)...")
search.fit(X_train, y_train)

Setting up hyperparameter search...
Using Recall@10 as optimization metric (optimizes for top-10 retrieval quality)

Starting hyperparameter search (this may take a while)...
Fitting 3 folds for each of 20 candidates, totalling 60 fits
Scaling features...
Scaling features...
Scaling features...
Scaling features...
Scaling features...
Scaling features...
Scaling features...
Scaling features...
Training CatBoost multi-output classifier...
  Input shape: (15929, 384)
  Output shape: (15929, 808)
Training CatBoost multi-output classifier...
  Input shape: (15929, 384)
  Output shape: (15929, 808)
Training CatBoost multi-output classifier...
  Input shape: (15929, 384)
  Output shape: (15929, 808)
Training CatBoost multi-output classifier...
  Input shape: (15930, 384)
  Output shape: (15930, 808)
Training CatBoost multi-output classifier...
  Input shape: (15929, 384)
  Output shape: (15929, 808)
Training CatBoost multi-output classifier...
  Input shape: (15930, 384)
  Output shape: (1593

In [ ]:
# Show best parameters
print("\n" + "="*70)
print("Best Parameters Found")
print("="*70)
for param, value in search.best_params_.items():
    print(f"{param}: {value}")

print(f"\nBest CV Score (Recall@10): {search.best_score_:.4f}")

In [ ]:
# Evaluate best model
print("\nEvaluating best model on test set...")
best_model = search.best_estimator_

y_pred_proba_tuned = best_model.predict_proba(X_test)
y_pred_tuned = best_model.predict(X_test, threshold=0.5)

results_cat_tuned = evaluate_term_prediction(
    y_test, y_pred_tuned, y_pred_proba_tuned, term_names, top_k=10
)

print_term_evaluation_results(results_cat_tuned)

 ## 6. Model Comparison

In [1]:
# Compare all models
comparison = pd.DataFrame([
    {
        'Model': 'CatBoost (No PCA)',
        'Micro F1': results_cat_no_pca['micro']['f1'],
        'Macro F1': results_cat_no_pca['macro']['f1'],
        'Recall@5': results_cat_no_pca['recall_at_k']['recall@5'],
        'Recall@10': results_cat_no_pca['recall_at_k']['recall@10'],
        'AUC Micro': results_cat_no_pca['auc']['micro'],
    },
    {
        'Model': 'CatBoost (With PCA)',
        'Micro F1': results_cat_pca['micro']['f1'],
        'Macro F1': results_cat_pca['macro']['f1'],
        'Recall@5': results_cat_pca['recall_at_k']['recall@5'],
        'Recall@10': results_cat_pca['recall_at_k']['recall@10'],
        'AUC Micro': results_cat_pca['auc']['micro'],
    },
    {
        'Model': 'CatBoost Tuned',
        'Micro F1': results_cat_tuned['micro']['f1'],
        'Macro F1': results_cat_tuned['macro']['f1'],
        'Recall@5': results_cat_tuned['recall_at_k']['recall@5'],
        'Recall@10': results_cat_tuned['recall_at_k']['recall@10'],
        'AUC Micro': results_cat_tuned['auc']['micro'],
    },
    {
        'Model': 'CatBoost Hierarchical (No PCA)',
        'Micro F1': results_cat_hier['micro']['f1'],
        'Macro F1': results_cat_hier['macro']['f1'],
        'Recall@5': results_cat_hier['recall_at_k']['recall@5'],
        'Recall@10': results_cat_hier['recall_at_k']['recall@10'],
        'AUC Micro': results_cat_hier['auc']['micro'],
    },
    {
        'Model': 'CatBoost Hierarchical (With PCA)',
        'Micro F1': results_cat_hier_pca['micro']['f1'],
        'Macro F1': results_cat_hier_pca['macro']['f1'],
        'Recall@5': results_cat_hier_pca['recall_at_k']['recall@5'],
        'Recall@10': results_cat_hier_pca['recall_at_k']['recall@10'],
        'AUC Micro': results_cat_hier_pca['auc']['micro'],
    }
])

print("\n" + "="*70)
print("CatBoost Model Comparison")
print("="*70)
print(comparison.to_string(index=False))

# Find best model
best_idx = comparison['Micro F1'].idxmax()
print(f"\n✅ Best model: {comparison.loc[best_idx, 'Model']} (F1={comparison.loc[best_idx, 'Micro F1']:.4f})")

NameError: name 'pd' is not defined

In [ ]:
# Visualize comparison
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# F1 scores
comparison[['Model', 'Micro F1', 'Macro F1']].set_index('Model').plot(kind='bar', ax=axes[0])
axes[0].set_title('F1 Scores Comparison')
axes[0].set_ylabel('F1 Score')
axes[0].legend(['Micro F1', 'Macro F1'])
axes[0].tick_params(axis='x', rotation=45)

# Recall@K
comparison[['Model', 'Recall@5', 'Recall@10']].set_index('Model').plot(kind='bar', ax=axes[1])
axes[1].set_title('Recall@K Comparison')
axes[1].set_ylabel('Recall')
axes[1].legend(['Recall@5', 'Recall@10'])
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 7. Save Models

In [ ]:
# Save models
print("Saving models...")

best_model.save('catboost_tuned_model.pkl')
cat_model_hier.save('catboost_hierarchical_model.pkl')

# Save PCA
import pickle
with open('pca_cogatlas_cat.pkl', 'wb') as f:
    pickle.dump(pca, f)

print("\n✅ Models saved!")
print("  - catboost_tuned_model.pkl")
print("  - catboost_hierarchical_model.pkl")
print("  - pca_cogatlas_cat.pkl")

In [ ]:
# Save results
save_term_results(results_cat_tuned, 'catboost_tuned_results.json')
save_term_results(results_cat_hier, 'catboost_hierarchical_results.json')

print("\n✅ Results saved!")

## Summary

**Key Findings:**
- CatBoost often achieves best overall performance
- Handles class imbalance automatically
- Robust to overfitting
- Good default parameters
- Works well with MultiOutputClassifier

**Recommendations:**
1. CatBoost is often the best choice for maximum accuracy
2. Start with hierarchical data for better class balance
3. Try both with and without PCA
4. Use hyperparameter tuning for production models
5. Monitor Recall@K for retrieval tasks